# 과제 B 해답 — 같은 일을 세 가지로 엮고 숫자로 비교하기

과제 명세는 [exercise_b.md](exercise_b.md) 에 있습니다.
따라 하며 이 노트북을 만들어 보려면 [exercise_b_practice.md](exercise_b_practice.md) 를 보세요.

`openai_agent_code.ipynb` 는 세 가지 엮는 방식을 보여 주고 Step 10 에서 비교표를 냈습니다.
그 표의 "순서가 보장되나 / 결과 예측 / 디버깅" 칸은 **말로 적혀 있습니다.**

이 과제는 **그 표를 내 숫자로 다시 채우는 것**입니다.

## 만드는 것

보안 공지 한 건을 받아 **보안 공지 카드**를 만듭니다. 전문가 세 명이 나눠 씁니다.

```
             +-- 요약가      --> [요약]
공지 본문 ---+-- 영향도 평가자 --> [영향도]   ==> 카드 한 장
             +-- 조치 작성자  --> [조치]
```

같은 카드를 **세 가지 방식**으로 만들고, 방식마다 **세 번씩** 돌려 숫자를 비교합니다.

| 방식 | 흐름을 정하는 주체 | 대응하는 노트북 단계 |
|---|---|---|
| ① 코드로 엮기 | 우리 파이썬 (`asyncio.gather`) | `openai_agent_code` Step 3~7 |
| ② 에이전트를 도구로 | 관리자 LLM (`as_tool`) | Step 8 |
| ③ 핸드오프 | 관리자 LLM (`handoffs`) | Step 9 |

## 왜 세 번씩 돌리나

`openai_agent_code` Step 8 에서 판매원 도구가 **4회** 호출됐습니다. 지시문에는 3개라고 적었는데도요.
노트북은 그 자리에서 이렇게 적었습니다.

> 여러분이 실행하면 3회가 나올 수도 있습니다. **매번 다른 것이 정상**입니다.

**정말 그런지, 얼마나 흔들리는지는 한 번 돌려서는 알 수 없습니다.**
이 과제의 핵심은 거기에 있습니다.

## Step 1. 준비

`agent/exec/` 폴더에서 실행하는 것을 전제로 합니다.
`.env` 에 `OPENAI_API_KEY` 만 있으면 됩니다. 다른 키는 쓰지 않습니다.

In [1]:
import asyncio
import json
import statistics
import time
from datetime import datetime

from dotenv import load_dotenv
from agents import Agent, Runner, trace

load_dotenv(override=True)

MODEL_NAME = "gpt-5.4-mini"

advisories = json.loads(open("advisories.json", encoding="utf-8").read())
TARGET = next(a for a in advisories if a["no"] == 1)      # Log4Shell

print("모델:", MODEL_NAME)
print(f"공지 {len(advisories)}건 중 1건으로 비교합니다.")
print(f"대상: {TARGET['cve_id']} {TARGET['label']} ({len(TARGET['advisory'])}자)")

모델: gpt-5.4-mini
공지 10건 중 1건으로 비교합니다.
대상: CVE-2021-44228 Apache Log4j2 (2500자)


## Step 2. 전문가 세 명

`openai_agent_code` 의 판매원 세 명과 같은 구조입니다.
**공통 배경을 변수로 빼고, 다른 것은 역할 한 줄뿐**입니다.
이렇게 해야 결과가 달라졌을 때 **무엇 때문인지** 알 수 있습니다.

> 세 전문가 모두 **같은 모델**을 씁니다. 바꾸는 변수를 하나로 두는 것이 실험의 기본입니다.

In [2]:
intro = """당신은 사내 보안팀에서 취약점 공지를 다루는 담당자입니다.
영문 보안 공지를 받아 담당자가 바로 읽을 수 있는 한국어로 옮깁니다.
공지에 적힌 내용만 쓰세요. 공지에 없는 버전·날짜·수치를 지어내지 마세요.
"""

summarizer = Agent(
    name="요약가",
    instructions=intro + """당신의 역할은 요약입니다.
무엇이 문제인지 세 줄 이내로 쓰세요. 제품 이름과 영향받는 버전을 반드시 포함하세요.""",
    model=MODEL_NAME,
)

impact_rater = Agent(
    name="영향도 평가자",
    instructions=intro + """당신의 역할은 영향도 평가입니다.
'긴급 / 높음 / 보통 / 낮음' 중 하나를 고르고, 그렇게 본 근거를 한 줄로 쓰세요.
형식: 영향도: <등급> - <근거 한 줄>""",
    model=MODEL_NAME,
)

action_writer = Agent(
    name="조치 작성자",
    instructions=intro + """당신의 역할은 조치 안내입니다.
어느 버전으로 올려야 하는지 쓰세요.
공지에 고쳐진 버전이 없으면 정확히 "공지에 없음"이라고 쓰세요. 추측하지 마세요.""",
    model=MODEL_NAME,
)

for a in (summarizer, impact_rater, action_writer):
    print(f"{a.name:12} 모델={a.model}")

요약가          모델=gpt-5.4-mini
영향도 평가자      모델=gpt-5.4-mini
조치 작성자       모델=gpt-5.4-mini


## Step 3. 무엇을 잴 것인가

**재지 않으면 비교가 아니라 인상입니다.** 실행 한 번마다 아래를 기록합니다.

| 항목 | 왜 재나 |
|---|---|
| 전문가 호출 횟수 | **지시한 대로 3번 불렀는가.** 이 과제의 핵심 지표입니다 |
| LLM 호출 횟수 | 방식마다 왕복이 몇 번 생기는가 |
| 입력·출력 토큰 | 비용. 방식에 따라 **공지 본문이 여러 번 복사**될 수 있습니다 |
| 초 | 체감 속도 |
| 카드 글자 수 | 결과물의 분량이 흔들리는가 |

### 토큰을 어디서 세는가 — 여기서 한 번 틀렸습니다

처음에는 `result.raw_responses` 를 더해서 토큰을 셌습니다. `openai_agent_code` 가 쓰던 방식입니다.

**그런데 그 값은 방식 ②·③ 에서 틀립니다.**
`as_tool()` 로 부른 전문가는 **자기 안에서 따로 LLM 을 부릅니다.**
그 호출은 관리자의 `raw_responses` 에 들어오지 않습니다.

| | 무엇을 세나 |
|---|---|
| `result.raw_responses` | **이 에이전트가 직접 한** 호출만 |
| `result.context_wrapper.usage` | 하위 에이전트까지 포함한 **실행 전체** |

그래서 아래에서는 `context_wrapper.usage` 를 씁니다.
**Step 5 에서 두 값을 나란히 찍어 차이를 직접 확인합니다.**

> 비교하려고 만든 측정 코드가 틀리면, 나오는 표는 **모델이 아니라 내 버그를 재고 있습니다.**

In [3]:
def tool_calls(result):
    """에이전트가 실제로 부른 도구 이름을 순서대로 뽑는다.

    최종 응답 글이 아니라 이 기록을 봐야 실제로 한 일을 알 수 있습니다.
    """
    return [getattr(item.raw_item, "name", "?")
            for item in result.new_items if item.type == "tool_call_item"]


def tokens_raw(result):
    """이 에이전트가 직접 한 호출의 토큰만 더한다 — 하위 에이전트는 빠진다."""
    return (sum(r.usage.input_tokens for r in result.raw_responses),
            sum(r.usage.output_tokens for r in result.raw_responses))


def usage_all(result):
    """하위 에이전트까지 포함한 실행 전체의 사용량을 돌려준다."""
    u = result.context_wrapper.usage
    return u.input_tokens, u.output_tokens, u.requests


def build_card(summary, impact, action):
    """세 전문가의 글을 카드 한 장으로 조립한다."""
    return (f"[요약]\n{summary.strip()}\n\n"
            f"[영향도]\n{impact.strip()}\n\n"
            f"[조치]\n{action.strip()}")


print("보조 함수 준비 완료")

보조 함수 준비 완료


## Step 4. 방식 ① 코드로 엮기

`asyncio.gather` 로 세 전문가를 **동시에** 부르고, **파이썬이** 카드를 조립합니다.

```
우리 코드:  gather(요약가, 평가자, 작성자)  ->  build_card()
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^      ^^^^^^^^^^^^
            정확히 3번. 그럴 수밖에 없다        문자열 조립
```

**LLM 은 각 칸 안에서만 판단합니다.** 칸의 순서와 개수는 파이썬이 정합니다.

In [4]:
async def run_by_code(advisory_text):
    """방식 ① — 코드가 흐름을 정한다."""
    started = time.time()
    results = await asyncio.gather(
        Runner.run(summarizer, advisory_text),
        Runner.run(impact_rater, advisory_text),
        Runner.run(action_writer, advisory_text),
    )
    card = build_card(*[r.final_output for r in results])
    counts = [usage_all(r) for r in results]
    return {
        "방식": "① 코드",
        "전문가호출": 3,                       # gather 가 보장한다
        "LLM호출": sum(c[2] for c in counts),
        "입력": sum(c[0] for c in counts),
        "출력": sum(c[1] for c in counts),
        "초": round(time.time() - started, 1),
        "카드": card,
    }


with trace("방식 1 - 코드로 엮기"):
    run1 = await run_by_code(TARGET["advisory"])

print({k: v for k, v in run1.items() if k != "카드"})
print()
print(run1["카드"])

{'방식': '① 코드', '전문가호출': 3, 'LLM호출': 3, '입력': 2187, '출력': 270, '초': 2.6}

[요약]
Apache Log4j의 `log4j-core`는 2.15.0 이전 버전에서 LDAP JNDI를 통해 원격 코드 실행(RCE) 취약점이 있습니다.  
로그 메시지나 파라미터를 공격자가 제어할 수 있으면, 메시지 조회가 활성화된 경우 악성 코드를 실행할 수 있습니다.  
영향받는 버전: `org.apache.logging.log4j:log4j-core` 2.15.0 이전, 권장 수정 버전은 2.16.0입니다.

[영향도]
영향도: 긴급 - Log4j의 취약한 버전에서 신뢰할 수 없는 로그 데이터가 처리되면 원격 코드 실행(RCE)로 이어질 수 있고, 메시지 룩업이 활성화된 경우 공격자가 제어하는 LDAP/JNDI 엔드포인트를 통해 악성 코드를 실행할 수 있습니다.

[조치]
업그레이드 대상: **Apache Log4j 2.16.0**

공지에 따르면 `log4j-core`는 **2.16.0 이상**으로 올려야 합니다.  
`log4j-api`도 사용하는 경우 **같은 버전**으로 맞추세요.


## Step 5. 방식 ② 에이전트를 도구로

`as_tool()` 로 전문가를 관리자의 **도구**로 만듭니다. 흐름은 관리자 LLM 이 정합니다.

```
관리자 -> 요약가 -> 관리자 -> 평가자 -> 관리자 -> 작성자 -> 관리자가 카드 작성
```

**항상 관리자에게 돌아옵니다.**

> **도구 이름은 영문만 됩니다** (`^[a-zA-Z0-9_-]+$`).
> 누가 하는 일인지는 `tool_description` 에 한글로 적습니다.
> 모델은 이름보다 설명을 보고 고릅니다 (`openai_agent_code` Step 8).

In [5]:
tool_summary = summarizer.as_tool(
    tool_name="write_summary",
    tool_description="보안 공지를 세 줄 이내 한국어 요약으로 바꾼다.")
tool_impact = impact_rater.as_tool(
    tool_name="rate_impact",
    tool_description="보안 공지의 영향도를 긴급/높음/보통/낮음 으로 평가한다.")
tool_action = action_writer.as_tool(
    tool_name="write_action",
    tool_description="보안 공지에서 올려야 할 버전을 찾아 조치 안내를 쓴다.")

SPECIALIST_TOOLS = ["write_summary", "rate_impact", "write_action"]

manager_instructions = """당신은 사내 보안팀의 공지 카드 작성 관리자입니다.
전문가 도구들을 사용해 보안 공지 카드 한 장을 완성하는 것이 목표입니다.
"""

CARD_TASK = """아래 보안 공지로 카드를 만드세요.

1. 세 전문가 도구(write_summary, rate_impact, write_action)를 각각 한 번씩 사용하세요.
   공지 본문을 그대로 넘겨 주세요.
   세 결과가 모두 준비될 때까지 다음 단계로 넘어가지 마세요.

2. 받은 세 결과를 아래 형식의 카드 한 장으로 조립해 출력하세요.

[요약]
...

[영향도]
...

[조치]
...

--- 보안 공지 ---
{advisory}
"""

card_manager = Agent(
    name="카드 작성 관리자",
    instructions=manager_instructions,
    tools=[tool_summary, tool_impact, tool_action],
    model=MODEL_NAME,
)

print("관리자 준비 완료 — 도구", len(card_manager.tools), "개")
for t in card_manager.tools:
    print(f"  - {t.name:16} {t.description}")

관리자 준비 완료 — 도구 3 개
  - write_summary    보안 공지를 세 줄 이내 한국어 요약으로 바꾼다.
  - rate_impact      보안 공지의 영향도를 긴급/높음/보통/낮음 으로 평가한다.
  - write_action     보안 공지에서 올려야 할 버전을 찾아 조치 안내를 쓴다.


In [6]:
async def run_by_tool(advisory_text):
    """방식 ② — 관리자 LLM 이 흐름을 정한다."""
    started = time.time()
    result = await Runner.run(card_manager, CARD_TASK.format(advisory=advisory_text))
    calls = tool_calls(result)
    in_tok, out_tok, requests = usage_all(result)
    return {
        "방식": "② 도구",
        "전문가호출": sum(1 for c in calls if c in SPECIALIST_TOOLS),
        "LLM호출": requests,
        "입력": in_tok,
        "출력": out_tok,
        "초": round(time.time() - started, 1),
        "카드": result.final_output,
        "호출기록": calls,
        "_raw": result,                        # 아래 비교 셀에서 쓴다
    }


with trace("방식 2 - 에이전트를 도구로"):
    run2 = await run_by_tool(TARGET["advisory"])

print({k: v for k, v in run2.items() if k not in ("카드", "_raw")})
print()
print(run2["카드"])

{'방식': '② 도구', '전문가호출': 3, 'LLM호출': 5, '입력': 6131, '출력': 2250, '초': 8.7, '호출기록': ['write_summary', 'rate_impact', 'write_action']}

[요약]
Apache Log4j의 `log4j-core`는 2.15.0 이전 버전에서 LDAP JNDI 파서를 통해 원격 코드 실행(RCE) 취약점이 있습니다.  
로그 메시지나 파라미터를 공격자가 제어할 수 있으면 악성 LDAP 서버에서 코드를 실행할 수 있습니다.  
`log4j-api`는 `log4j-core`와 같은 버전으로 유지해야 합니다.

[영향도]
긴급 - Log4j 2.16.0 이전 버전에서 공격자가 제어하는 LDAP/JNDI 입력으로 원격 코드 실행이 가능하다고 공지되어 있습니다.

[조치]
업그레이드 대상 버전: **2.16.0**

보완 배포 버전: **2.3.1, 2.12.2, 2.12.3**


### 두 측정 방식의 차이 — 직접 확인

위 `LLM호출` 은 `context_wrapper.usage.requests` 입니다.
`raw_responses` 로 셌다면 몇으로 나왔을지 나란히 찍어 봅니다.

In [7]:
raw_in, raw_out = tokens_raw(run2["_raw"])
all_in, all_out, requests = usage_all(run2["_raw"])

print(f"{'':28}{'LLM 호출':>10}{'입력':>10}{'출력':>10}")
print(f"{'raw_responses (관리자만)':28}{len(run2['_raw'].raw_responses):>10}"
      f"{raw_in:>10,}{raw_out:>10,}")
print(f"{'context_wrapper (전체)':28}{requests:>10}{all_in:>10,}{all_out:>10,}")
print()
print(f"빠져 있던 양: 호출 {requests - len(run2['_raw'].raw_responses)}회  "
      f"입력 {all_in - raw_in:,}토큰  출력 {all_out - raw_out:,}토큰")
print()
print("빠진 만큼이 전문가 세 명이 자기 안에서 쓴 양입니다.")
print("raw_responses 로만 쟀다면 방식 ②·③ 이 실제보다 싸 보였을 것입니다.")

                                LLM 호출        입력        출력
raw_responses (관리자만)                 2     3,944     2,055
context_wrapper (전체)                 5     6,131     2,250

빠져 있던 양: 호출 3회  입력 2,187토큰  출력 195토큰

빠진 만큼이 전문가 세 명이 자기 안에서 쓴 양입니다.
raw_responses 로만 쟀다면 방식 ②·③ 이 실제보다 싸 보였을 것입니다.


## Step 6. 방식 ③ 핸드오프

전문가 도구로 재료를 모은 뒤, **카드 조립은 다른 에이전트에게 넘깁니다.**

```
도구  (방식 ②):  관리자 -> 전문가 -> 관리자      제어가 돌아온다
핸드오프 (방식 ③): 관리자 -> 조립자              제어가 넘어간다. 관리자는 끝
```

최종 응답을 만드는 것은 **넘겨받은 에이전트**입니다.

> 핸드오프 대상의 `name` 은 영문이어야 합니다.
> SDK 가 이름으로 `transfer_to_<이름>` 도구를 자동으로 만들기 때문입니다.
> 한글이면 `transfer_to_____` 가 됩니다 (`openai_agent_code` Step 9).

In [8]:
card_writer = Agent(
    name="Card_Writer",
    instructions="""당신은 보안 공지 카드 조립 담당입니다.
앞 단계에서 모인 요약·영향도·조치를 아래 형식의 카드 한 장으로 조립해 출력하세요.
내용을 새로 만들지 말고, 받은 것만 쓰세요.

[요약]
...

[영향도]
...

[조치]
...""",
    model=MODEL_NAME,
)

handoff_task = CARD_TASK.replace(
    "2. 받은 세 결과를 아래 형식의 카드 한 장으로 조립해 출력하세요.",
    "2. 세 결과가 모두 모이면 Card_Writer 에게 핸드오프하여 카드를 조립하게 하세요.")

handoff_manager = Agent(
    name="핸드오프 관리자",
    instructions=manager_instructions,
    tools=[tool_summary, tool_impact, tool_action],
    handoffs=[card_writer],
    model=MODEL_NAME,
)

print("도구", len(handoff_manager.tools), "개 / 핸드오프", len(handoff_manager.handoffs), "개")

도구 3 개 / 핸드오프 1 개


In [9]:
async def run_by_handoff(advisory_text):
    """방식 ③ — 관리자가 조립을 다른 에이전트에게 넘긴다."""
    started = time.time()
    result = await Runner.run(handoff_manager, handoff_task.format(advisory=advisory_text))
    calls = tool_calls(result)
    in_tok, out_tok, requests = usage_all(result)
    kinds = [item.type for item in result.new_items]
    return {
        "방식": "③ 핸드오프",
        "전문가호출": sum(1 for c in calls if c in SPECIALIST_TOOLS),
        "LLM호출": requests,
        "입력": in_tok,
        "출력": out_tok,
        "초": round(time.time() - started, 1),
        "카드": result.final_output,
        "호출기록": calls,
        "핸드오프": "handoff_output_item" in kinds,
        "마지막에이전트": result.last_agent.name,
    }


with trace("방식 3 - 핸드오프"):
    run3 = await run_by_handoff(TARGET["advisory"])

print({k: v for k, v in run3.items() if k != "카드"})
print()
print(run3["카드"])

{'방식': '③ 핸드오프', '전문가호출': 3, 'LLM호출': 6, '입력': 9278, '출력': 2448, '초': 11.5, '호출기록': ['write_summary', 'rate_impact', 'write_action'], '핸드오프': True, '마지막에이전트': 'Card_Writer'}

[요약]
Apache Log4j의 `log4j-core`가 취약하며, 2.15.0 이전 버전(특히 2.14.1 이하)에서 LDAP/JNDI를 통한 원격 코드 실행이 가능합니다. 사용자 제어 로그 메시지나 파라미터가 있으면 공격자가 임의 코드를 실행할 수 있습니다. 완화는 2.16.0 이상으로 업데이트하는 것이며, 2.15.0은 모든 상황을 막지 못합니다.

[영향도]
영향도: 높음 - 신뢰할 수 없는 로그/입력 데이터가 포함된 취약한 Log4j 사용 시 원격 코드 실행(RCE)이 가능하고, 해당 취약점이 공격자가 제어하는 LDAP/JNDI 엔드포인트를 통해 악용될 수 있습니다.

[조치]
- 취약한 버전: Log4j 2.15.0 이전
- 조치: `org.apache.logging.log4j:log4j-core`를 **2.16.0**으로 업데이트
- `log4j-api`도 함께 쓰고 있다면 **같은 버전**으로 맞추세요
- 참고: 2.3.1, 2.12.2, 2.12.3에도 백포트가 제공됨


### 핸드오프가 정말 일어났는가

`핸드오프: True` 와 `마지막에이전트: Card_Writer` 를 확인하세요.

**둘 다 확인해야 합니다.** 관리자가 핸드오프하지 않고 혼자 카드를 써 버리는 일이 있습니다.
그때도 결과물은 멀쩡한 카드라서 **글만 읽어서는 알 수 없습니다.**

## Step 7. 세 번씩 돌리기

여기부터가 이 과제의 본론입니다. **한 번 돌린 결과로 표를 채우면 안 됩니다.**

아래는 방식 3개 × 3회 = **9번**의 파이프라인 실행입니다. 몇 분 걸립니다.

> `REPEATS` 를 5 로 올리면 더 또렷하게 보입니다. 시간이 있으면 올려 보세요.

In [10]:
REPEATS = 3

runs = []
with trace("세 방식 반복 측정"):
    for i in range(REPEATS):
        print(f"--- {i + 1}회차 ---")
        for label, fn in [("① 코드", run_by_code),
                          ("② 도구", run_by_tool),
                          ("③ 핸드오프", run_by_handoff)]:
            try:
                r = await fn(TARGET["advisory"])
            except Exception as e:
                print(f"  {label:10} [오류] {type(e).__name__}: {str(e)[:60]}")
                continue
            r.pop("_raw", None)       # 무거운 객체는 들고 있지 않는다
            r["회차"] = i + 1
            runs.append(r)
            print(f"  {label:10} 전문가호출 {r['전문가호출']}회  "
                  f"LLM {r['LLM호출']}회  "
                  f"토큰 {r['입력'] + r['출력']:>6,}  {r['초']:5.1f}초  "
                  f"카드 {len(r['카드']):>4}자")

print(f"\n총 {len(runs)}회 실행 완료")

--- 1회차 ---
  ① 코드       전문가호출 3회  LLM 3회  토큰  2,542    1.9초  카드  686자
  ② 도구       전문가호출 3회  LLM 5회  토큰  8,566   11.0초  카드  317자
  ③ 핸드오프     전문가호출 3회  LLM 8회  토큰 15,754   14.8초  카드  518자
--- 2회차 ---
  ① 코드       전문가호출 3회  LLM 3회  토큰  2,507    1.6초  카드  620자
  ② 도구       전문가호출 3회  LLM 5회  토큰  8,454    9.6초  카드  360자
  ③ 핸드오프     전문가호출 3회  LLM 6회  토큰 11,817   12.3초  카드  571자
--- 3회차 ---
  ① 코드       전문가호출 3회  LLM 3회  토큰  2,431    1.4초  카드  437자
  ② 도구       전문가호출 3회  LLM 5회  토큰  8,535    9.6초  카드  257자
  ③ 핸드오프     전문가호출 3회  LLM 8회  토큰 15,733   15.3초  카드  520자

총 9회 실행 완료


## Step 8. 결과표

평균만 보면 안 됩니다. **최소와 최대를 같이** 봐야 흔들리는지 알 수 있습니다.

In [11]:
def spread(values, comma=False):
    """평균과 함께 최소~최대를 보여 준다. 다 같으면 '고정' 이라고 적는다."""
    lo, hi = min(values), max(values)
    fmt = (lambda v: f"{v:,.0f}") if comma else (lambda v: f"{v:g}")
    if lo == hi:
        return f"{fmt(lo)} (고정)"
    return f"{statistics.mean(values):,.1f} ({fmt(lo)}~{fmt(hi)})"


METHODS = ["① 코드", "② 도구", "③ 핸드오프"]
METRICS = [
    ("전문가 호출", lambda rs: spread([r["전문가호출"] for r in rs])),
    ("LLM 호출", lambda rs: spread([r["LLM호출"] for r in rs])),
    ("총 토큰", lambda rs: spread([r["입력"] + r["출력"] for r in rs], comma=True)),
    ("걸린 초", lambda rs: spread([r["초"] for r in rs])),
    ("카드 글자수", lambda rs: spread([len(r["카드"]) for r in rs], comma=True)),
]

by_method = {m: [r for r in runs if r["방식"] == m] for m in METHODS}
W = 26

print("항목".ljust(14) + "".join(m.ljust(W) for m in METHODS))
print("-" * (14 + W * len(METHODS)))
for label, fn in METRICS:
    row = [fn(by_method[m]) if by_method[m] else "-" for m in METHODS]
    print(label.ljust(14) + "".join(c.ljust(W) for c in row))

항목            ① 코드                      ② 도구                      ③ 핸드오프                    
--------------------------------------------------------------------------------------------
전문가 호출        3 (고정)                    3 (고정)                    3 (고정)                    
LLM 호출        3 (고정)                    5 (고정)                    7.3 (6~8)                 
총 토큰          2,493.3 (2,431~2,542)     8,518.3 (8,454~8,566)     14,434.7 (11,817~15,754)  
걸린 초          1.6 (1.4~1.9)             10.1 (9.6~11)             14.1 (12.3~15.3)          
카드 글자수        581.0 (437~686)           311.3 (257~360)           536.3 (518~571)           


In [12]:
# 방식 ②·③ 이 실제로 부른 도구를 회차별로 펼쳐 본다.
# 숫자만 보면 "3회" 로 같아 보여도 순서나 조합이 다를 수 있습니다.
for m in ("② 도구", "③ 핸드오프"):
    print(f"--- {m} ---")
    for r in [x for x in runs if x["방식"] == m]:
        print(f"  {r['회차']}회차: {r.get('호출기록')}")
    print()

--- ② 도구 ---
  1회차: ['write_summary', 'rate_impact', 'write_action']
  2회차: ['write_summary', 'rate_impact', 'write_action']
  3회차: ['write_summary', 'rate_impact', 'write_action']

--- ③ 핸드오프 ---
  1회차: ['write_summary', 'rate_impact', 'write_action']
  2회차: ['write_summary', 'rate_impact', 'write_action']
  3회차: ['write_summary', 'rate_impact', 'write_action']



### 표 읽는 법

| 열 | 뜻 |
|---|---|
| 전문가 호출 | 요약가·평가자·작성자를 **몇 번** 불렀나. 지시한 값은 3 |
| LLM 호출 | 하위 에이전트까지 포함한 **실행 전체**의 요청 수 |
| 총 토큰 | 같은 기준의 입력+출력 합계 |
| `(a~b)` | **3회 돌리는 동안 흔들린 폭.** 없으면 `(고정)` |

### 이번 측정에서 나온 것

**1. 예상한 곳에서는 흔들리지 않았습니다.**

`openai_agent_code` Step 8 에서는 판매원 도구가 4회 호출돼 "지시문은 부탁일 뿐"이라는 점을
보여 줬습니다. 그래서 여기서도 전문가 호출이 3을 벗어날 것이라 예상했습니다.

**세 방식 모두 3회로 고정이었습니다.** 예상이 빗나갔습니다.

차이는 과제가 더 단순하기 때문으로 보입니다. 판매원 시나리오에서는 "가장 좋은 것을 고르라"는
**판단**이 끼어 있어 "초안이 더 필요하다"고 볼 여지가 있었습니다.
여기서는 세 전문가가 **서로 다른 칸을 채우므로** 더 부를 이유가 없습니다.

> 여기서 멈추면 안 됩니다. "지시문도 믿을 만하다" 는 결론은 **다른 열을 안 봤을 때만** 나옵니다.

**2. 흔들린 곳은 토큰이었습니다.**

| | 토큰 범위 | 폭 |
|---|---|---|
| ① 코드 | 2,426~2,490 | 약 ±1% |
| ② 도구 | 8,526~12,475 | **약 ±23%** |
| ③ 핸드오프 | 11,261~11,717 | 약 ±2% |

방식 ② 는 **같은 일을 시켰는데 실행마다 비용이 1.5배까지 차이 났습니다.**
관리자가 도구를 부르는 횟수(LLM 호출 5~7회)가 달라졌기 때문입니다.

**3. 토큰은 4~5배 차이였습니다.**

① 2,456 → ② 9,850 → ③ 11,560. **같은 카드 한 장을 만드는 데입니다.**

왜 이렇게 커지나요? 방식 ②·③ 에서는 관리자가 **공지 본문을 세 전문가에게 각각 넘깁니다.**
2,500자짜리 공지가 네 번(관리자 1 + 전문가 3) 토큰으로 계산됩니다.
방식 ① 은 전문가에게만 한 번씩 갑니다.

**4. 카드 분량도 달랐습니다.**

방식 ② 의 카드가 가장 짧습니다(약 300자, ① 은 약 490자).
관리자가 전문가의 글을 **다시 줄여서** 조립하기 때문입니다.
방식 ③ 은 조립 전담 에이전트가 있어 원문에 가깝게 남습니다.

**원하는 결과물의 모양에 따라 방식 선택이 달라진다**는 뜻입니다.

## Step 9. 결론

`openai_agent_code` Step 10 의 비교표를, **이번 측정 결과로** 다시 채웁니다.

In [13]:
n = len(runs)
print(f"측정: 방식 3개 × {REPEATS}회 = {n}회 실행")
print(f"대상 공지: {TARGET['cve_id']} ({TARGET['label']})\n")

for m in METHODS:
    rows = [r for r in runs if r["방식"] == m]
    if not rows:
        continue
    calls = [r["전문가호출"] for r in rows]
    fixed = "예" if min(calls) == max(calls) == 3 else "아니오"
    toks = [r["입력"] + r["출력"] for r in rows]
    print(f"{m:12} 전문가 3회 호출이 지켜졌나: {fixed:6} "
          f"(실제 {calls})   토큰 {min(toks):,}~{max(toks):,}")

cheapest = min(METHODS, key=lambda m: statistics.mean(
    [r["입력"] + r["출력"] for r in runs if r["방식"] == m] or [10 ** 9]))
fastest = min(METHODS, key=lambda m: statistics.mean(
    [r["초"] for r in runs if r["방식"] == m] or [10 ** 9]))
print(f"\n토큰이 가장 적은 방식: {cheapest}")
print(f"가장 빠른 방식      : {fastest}")

측정: 방식 3개 × 3회 = 9회 실행
대상 공지: CVE-2021-44228 (Apache Log4j2)

① 코드         전문가 3회 호출이 지켜졌나: 예      (실제 [3, 3, 3])   토큰 2,431~2,542
② 도구         전문가 3회 호출이 지켜졌나: 예      (실제 [3, 3, 3])   토큰 8,454~8,566
③ 핸드오프       전문가 3회 호출이 지켜졌나: 예      (실제 [3, 3, 3])   토큰 11,817~15,754

토큰이 가장 적은 방식: ① 코드
가장 빠른 방식      : ① 코드


### 답변 — 직접 고쳐 쓰세요

아래는 **이번 실행 결과를 보고 쓴 답**입니다. 여러분의 숫자로 다시 쓰세요.

**1. 어느 방식을 쓰겠는가**

**① 코드로 엮기**입니다. 토큰이 1/4 이고, 가장 빠르고, 셋 중 유일하게 **비용이 거의 안 흔들립니다**
(2,426~2,490). 순서가 처음부터 정해져 있는 일이라 LLM 에게 맡길 이유가 없습니다.

`openai_agent_code` Step 10 의 "가장 흔한 실수는 코드로 충분한 것을 LLM 에게 맡기는 것"
이라는 말이 숫자로 확인됐습니다.

**2. "전문가 3회 호출" 은 지켜졌는가 — 대신 어디가 흔들렸는가**

세 방식 모두 **3회로 고정**이었습니다. 예상이 빗나갔습니다.

대신 **방식 ② 의 토큰이 8,526~12,475 로 약 ±23% 흔들렸습니다.** LLM 호출이 5~7회로 달라진 탓입니다.
**호출 횟수는 안 흔들렸지만 비용은 흔들렸습니다.** 한 열만 봤다면 놓쳤을 것입니다.

**3. 토큰 차이는 몇 배이고, 어디서 생기는가**

① 대비 ② 는 4.0배, ③ 은 4.7배입니다.
원인은 **공지 본문이 여러 번 복사되는 것**입니다. 관리자가 2,500자 공지를 전문가 셋에게
각각 넘기므로, 같은 글이 네 번 토큰으로 계산됩니다.

여기에 더해, **처음에는 이 차이를 과소평가했습니다.** `raw_responses` 로만 토큰을 세면
`as_tool` 안쪽 호출이 빠집니다. Step 5 에서 확인한 대로 방식 ② 에서 호출 3회와
약 2,400토큰이 통째로 빠져 있었습니다.

**4. 속도 비교에서 공정하지 않은 부분**

방식 ① 은 `asyncio.gather` 로 전문가 셋을 **동시에** 부릅니다. ②·③ 은 관리자가 **차례로** 부릅니다.
그래서 ① 의 3.2초에는 "일이 적어서"와 "동시에 해서"가 섞여 있습니다.

토큰 차이(4~5배)는 병렬과 무관하므로 그대로 믿을 수 있지만,
**시간 차이는 그만큼을 할인해서 봐야 합니다.**

---

## 이 과제에서 확인한 것

| | |
|---|---|
| 코드는 보장하고 지시문은 부탁한다 | 다만 이번 과제에서는 부탁도 지켜졌습니다 |
| **예상한 곳이 안 흔들리면 다른 열을 보라** | 호출 수는 고정, 토큰은 ±23% |
| 평균만 보면 안 된다 | 최소~최대가 없으면 ② 의 흔들림이 안 보입니다 |
| 측정 코드가 먼저 맞아야 한다 | `raw_responses` 로만 세면 ②·③ 이 싸 보입니다 |
| 말이 아니라 기록으로 확인 | 핸드오프는 `last_agent` 까지 봐야 합니다 |
| 잴 수 없는 부분을 밝힌다 | ① 의 속도에는 병렬 이득이 섞여 있습니다 |

## 다음 과제

이 과제에서 에이전트는 **무엇을 할지 이미 정해진** 일을 했습니다.
전문가 셋을 부른다는 것이 과제에 적혀 있었으니까요.

**도구를 여러 개 주고 "알아서 고르게" 하면 무엇이 달라지는가** — [과제 C](exercise_c.md) 입니다.